In [ ]:
import os
import random
import time
from dataclasses import dataclass
from typing import Optional
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import tyro
from torch.distributions.normal import Normal
from torch.utils.tensorboard import SummaryWriter

import warnings
warnings.filterwarnings('ignore')

## Hyperparameters

In [ ]:
@dataclass
class Args:
    exp_name: str = "inverted_pendulum"
    """the name of this experiment"""
    seed: int = 1
    """seed of the experiment"""
    torch_deterministic: bool = True
    """if toggled, `torch.backends.cudnn.deterministic=False`"""
    cuda: bool = False #True
    """if toggled, cuda will be enabled by default"""
    capture_video: bool = False
    """whether to capture videos of the agent performances (check out `videos` folder)"""
    save_model: bool = True
    """whether to save model into the `runs/{run_name}` folder"""

    # Algorithm specific arguments
    env_id: str = "InvertedPendulum-v0"
    """the id of the environment"""
    total_timesteps: int = 500000
    """total timesteps of the experiments"""
    learning_rate: float = 3e-4
    """the learning rate of the optimizer"""
    num_envs: int = 1
    """the number of parallel game environments"""
    num_steps: int = 3000
    """the number of steps to run in each environment per policy rollout"""
    anneal_lr: bool = True
    """Toggle learning rate annealing for policy and value networks"""
    gamma: float = 0.995
    """the discount factor gamma"""
    gae_lambda: float = 0.98
    """the lambda for the general advantage estimation"""
    num_minibatches: int = 10
    """the number of mini-batches"""
    update_epochs: int = 10
    """the K epochs to update the policy"""
    norm_adv: bool = True
    """Toggles advantages normalization"""
    clip_coef: float = 0.2
    """the surrogate clipping coefficient"""
    clip_vloss: bool = True
    """Toggles whether or not to use a clipped loss for the value function, as per the paper."""
    ent_coef: float = 0.0
    """coefficient of the entropy"""
    vf_coef: float = 0.5
    """coefficient of the value function"""
    max_grad_norm: float = 0.5
    """the maximum norm for the gradient clipping"""
    target_kl: Optional[float] = None
    """the target KL divergence threshold"""
    normalize: bool = False
    """use normalization (observation/reward)"""
    checkpoint_freq: int=50
    """save checkpoint every N iterations"""
    
    # For the runtime
    batch_size: int = 0
    """the batch size (computed in runtime)"""
    minibatch_size: int = 0
    """the mini-batch size (computed in runtime)"""
    num_iterations: int = 0
    """the number of iterations (computed in runtime)"""

## Environment

In [ ]:
class InvertedPendulumEnv(gym.Wrapper):
    def __init__(self):
        env = gym.make('Pendulum-v1', max_episode_steps=300)
        super().__init__(env)
        self.unwrapped.max_torque = 15.
        self.unwrapped.max_speed = 60.
        self.unwrapped.action_space = gym.spaces.Box(low=-self.unwrapped.max_torque, high=self.unwrapped.max_torque, shape=(1,))
        high = np.array([1., 1., self.unwrapped.max_speed])
        self.unwrapped.observation_space = gym.spaces.Box(low=-high, high=high)
        self.action_space = self.unwrapped.action_space
        self.observation_space = self.unwrapped.observation_space

    def reset(self, **kwargs):
        while True:
            obs, info = self.env.reset(**kwargs)
            if abs(self.unwrapped.state[0]) <= 1.0:
                return obs, info
            kwargs.pop('seed', None)

gym.register(id="InvertedPendulum-v0", entry_point=lambda: InvertedPendulumEnv())


def make_env(env_id, idx, capture_video, run_name, gamma, normalize=True):
    def thunk():
        if capture_video and idx == 0:
            env = gym.make(env_id, render_mode="rgb_array")
            env = gym.wrappers.RecordVideo(env, f"videos/{run_name}")
        else:
            env = gym.make(env_id)

        env = gym.wrappers.FlattenObservation(env)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env = gym.wrappers.ClipAction(env)

        if normalize:
            env = gym.wrappers.NormalizeObservation(env)
            env = gym.wrappers.TransformObservation(env, lambda obs: np.clip(obs, -10, 10))
            env = gym.wrappers.NormalizeReward(env, gamma=gamma)
            env = gym.wrappers.TransformReward(env, lambda reward: np.clip(reward, -10, 10))

        return env
    return thunk

## Agent: Actor-Critic (Neural Networks)

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

class Agent(nn.Module):
    def __init__(self, envs):
        super().__init__()
        # Critic
        # Input: State
        # Output: V(s)
        self.critic = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 1), std=1.0),
        )

        # Actor
        # Input: State
        # Output: Action_Mean(s)
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(np.array(envs.single_observation_space.shape).prod(), 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, 64)),
            nn.Tanh(),
            layer_init(nn.Linear(64, np.prod(envs.single_action_space.shape)), std=0.01),
        )
        self.actor_logstd = nn.Parameter(torch.zeros(1, np.prod(envs.single_action_space.shape)))

    def get_value(self, x):
        return self.critic(x)

    # Sample action and compute V(s)
    def get_action_and_value(self, x, action=None):
        action_mean = self.actor_mean(x)
        action_logstd = self.actor_logstd.expand_as(action_mean)
        action_std = torch.exp(action_logstd)
        probs = Normal(action_mean, action_std)
        if action is None:
            action = probs.sample()
        return action, probs.log_prob(action).sum(1), probs.entropy().sum(1), self.critic(x)

## Experiment Arguments and Logs settings

In [ ]:
args = Args()
args.batch_size = int(args.num_envs * args.num_steps)
args.minibatch_size = int(args.batch_size // args.num_minibatches)
args.num_iterations = args.total_timesteps // args.batch_size
run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

# CSV Logging setup
csv_path_ppo = f"runs/{run_name}/log_episodes_ppo.csv"
log_episodes_ppo = []

writer = SummaryWriter(f"runs/{run_name}")
writer.add_text(
    "hyperparameters",
    "|param|value|\n|-|-|\n%s" % ("\n".join([f"|{key}|{value}|" for key, value in vars(args).items()])),
)

## Environment Setup

In [ ]:
# Seeding
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

device = torch.device("cuda" if torch.cuda.is_available() and args.cuda else "cpu")

# Environment setup
envs = gym.vector.SyncVectorEnv(
    [make_env(args.env_id, i, args.capture_video, run_name, args.gamma, normalize=args.normalize) for i in range(args.num_envs)]
)
assert isinstance(envs.single_action_space, gym.spaces.Box), "only continuous action space is supported"

agent = Agent(envs).to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate, eps=1e-5)

## Storage Setup and Training Initialization

In [ ]:
# Storage Setup
obs = torch.zeros((args.num_steps, args.num_envs) + envs.single_observation_space.shape).to(device)
actions = torch.zeros((args.num_steps, args.num_envs) + envs.single_action_space.shape).to(device)
logprobs = torch.zeros((args.num_steps, args.num_envs)).to(device)
rewards = torch.zeros((args.num_steps, args.num_envs)).to(device)
dones = torch.zeros((args.num_steps, args.num_envs)).to(device)
values = torch.zeros((args.num_steps, args.num_envs)).to(device)

episode_theta_list = []
episode_safe_list = []
ep_max_angle = 0.0
ep_violations = 0

# Initialization
global_step = 0
start_time = time.time()
next_obs, _ = envs.reset(seed=args.seed)
next_obs = torch.Tensor(next_obs).to(device)
next_done = torch.zeros(args.num_envs).to(device)

## Training

In [ ]:

for iteration in range(1, args.num_iterations + 1):
    # Learning rate Annealing
    if args.anneal_lr:
        frac = 1.0 - (iteration - 1.0) / args.num_iterations
        lrnow = frac * args.learning_rate
        optimizer.param_groups[0]["lr"] = lrnow

    ###########################################
    # Rollout and collect data
    ###########################################
    for step in range(0, args.num_steps):
        global_step += args.num_envs
        obs[step] = next_obs
        dones[step] = next_done

        # Sample action from policy
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(next_obs)
            values[step] = value.flatten()
        actions[step] = action
        logprobs[step] = logprob

        # Track angle
        obs_np = next_obs[0].cpu().numpy()
        theta_now = np.arctan2(obs_np[1], obs_np[0])
        omega_now = obs_np[2]
        episode_theta_list.append(abs(np.degrees(theta_now)))
        episode_safe_list.append(int(abs(theta_now) + 0.01 * abs(omega_now) > 1.0))

        # Agent (step) in environment
        next_obs, reward, terminations, truncations, infos = envs.step(action.cpu().numpy())
        next_done = np.logical_or(terminations, truncations)
        rewards[step] = torch.tensor(reward).to(device).view(-1)
        next_obs, next_done = torch.Tensor(next_obs).to(device), torch.Tensor(next_done).to(device)

        # Logging
        if "final_info" in infos:
            for info in infos["final_info"]:
                if info and "episode" in info:
                    ep_r = info["episode"]["r"].item()
                    ep_l = info["episode"]["l"].item()

                    ep_max_angle  = max(episode_theta_list) if episode_theta_list else 0.0
                    ep_violations = int(sum(episode_safe_list)) if episode_safe_list else 0
                    episode_theta_list = []
                    episode_safe_list  = []

                    print(f"global_step={global_step}, episodic_return={ep_r:.1f}", flush=True)
                    writer.add_scalar("charts/episodic_return", ep_r, global_step)
                    writer.add_scalar("charts/episodic_length", ep_l, global_step)

                    # CSV logs
                    log_episodes_ppo.append({
                        "global_step": global_step,
                        "episode_return": round(ep_r, 4),
                        "episode_length": int(ep_l),
                        "max_angle_deg": round(ep_max_angle, 3),
                        "safety_violations": ep_violations,
                    })

    ###########################################
    # Compute GAE
    ###########################################
    with torch.no_grad():
        next_value = agent.get_value(next_obs).reshape(1, -1)
        advantages = torch.zeros_like(rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(args.num_steps)):
            if t == args.num_steps - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
            delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
            advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
        returns = advantages + values

    # Flatten the batch
    b_obs = obs.reshape((-1,) + envs.single_observation_space.shape)
    b_logprobs = logprobs.reshape(-1)
    b_actions = actions.reshape((-1,) + envs.single_action_space.shape)
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = values.reshape(-1)

    ###########################################    
    # Optimizing the Policy and Value network
    ###########################################
    b_inds = np.arange(args.batch_size)
    clipfracs = []
    for epoch in range(args.update_epochs):
        np.random.shuffle(b_inds)

        # Batch iteration
        for start in range(0, args.batch_size, args.minibatch_size):
            # Minibatch
            end = start + args.minibatch_size
            mb_inds = b_inds[start:end]

            # Policies Ratio
            _, newlogprob, entropy, newvalue = agent.get_action_and_value(b_obs[mb_inds], b_actions[mb_inds])
            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            # KL divergance approx.
            with torch.no_grad():
                old_approx_kl = (-logratio).mean()
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs += [((ratio - 1.0).abs() > args.clip_coef).float().mean().item()]

            # Advantage normalization
            mb_advantages = b_advantages[mb_inds]
            if args.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            # Policy loss (Clipped PPO)
            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - args.clip_coef, 1 + args.clip_coef)
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            # Value loss
            newvalue = newvalue.view(-1)
            if args.clip_vloss:
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds],
                    -args.clip_coef,
                    args.clip_coef,
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss_max = torch.max(v_loss_unclipped, v_loss_clipped)
                v_loss = 0.5 * v_loss_max.mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

            # Entropy
            entropy_loss = entropy.mean()

            # Total loss
            loss = pg_loss - args.ent_coef * entropy_loss + v_loss * args.vf_coef

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), args.max_grad_norm)
            optimizer.step()

        # Early stopping
        if args.target_kl is not None and approx_kl > args.target_kl:
            break

    # Logging
    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    # Record rewards (plotting purposes)
    writer.add_scalar("charts/learning_rate", optimizer.param_groups[0]["lr"], global_step)
    writer.add_scalar("losses/value_loss", v_loss.item(), global_step)
    writer.add_scalar("losses/policy_loss", pg_loss.item(), global_step)
    writer.add_scalar("losses/entropy", entropy_loss.item(), global_step)
    writer.add_scalar("losses/old_approx_kl", old_approx_kl.item(), global_step)
    writer.add_scalar("losses/approx_kl", approx_kl.item(), global_step)
    writer.add_scalar("losses/clipfrac", np.mean(clipfracs), global_step)
    writer.add_scalar("losses/explained_variance", explained_var, global_step)
    writer.add_scalar("charts/SPS", int(global_step / (time.time() - start_time)), global_step)

## Save model

In [ ]:
if args.save_model:
    model_path = f"runs/{run_name}/{args.exp_name}.pt"
    torch.save(agent.state_dict(), model_path)
    print(f"model saved to {model_path}")

envs.close()
writer.close()

pd.DataFrame(log_episodes_ppo).to_csv(csv_path_ppo, index=False)
print(f"Logs saved to {csv_path_ppo}")

## Plots

In [ ]:
ep = pd.read_csv(csv_path_ppo)

def smooth(s, w=20):
    return s.rolling(w, min_periods=1, center=True).mean()

def plot_ep(ax, col, title, ylabel, limit=None):
    ax.plot(ep.global_step, ep[col], alpha=0.2, linewidth=0.8)
    ax.plot(ep.global_step, smooth(ep[col]), linewidth=2)
    if limit:
        ax.axhline(limit, color="red", linestyle="--", linewidth=1.2, label="safe limit")
        ax.legend(fontsize=8)
    ax.set_title(title)
    ax.set_xlabel("Step")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=4))

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
fig.suptitle("PPO Training", fontsize=13)
fig.subplots_adjust(hspace=0.4, wspace=0.35)

plot_ep(axes[0,0], "episode_return", "Episode Return", "Return")
plot_ep(axes[0,1], "episode_length", "Episode Length", "Steps")
plot_ep(axes[1,0], "max_angle_deg", "Max Angle per Episode", "Degrees", limit=np.degrees(1.0))
plot_ep(axes[1,1], "safety_violations", "Safety Violations", "Steps")

fig.savefig(f"runs/{run_name}/training_curves_ppo.png", bbox_inches="tight")
plt.show()